# Evaluating Currently Available Free-Tier Reasoning Large Language Models on TruthfulQA

---

## Table of Contents

- [Prerequisites](#prerequisites)
- [Research Question](#research-question)
- [Dataset](#dataset)
    - [Description](#description)
    - [Data Collection](#data-collection)
    - [Structure](#structure)
- [Data Cleaning](#data-cleaning)
    - [Response](#response)
    - [Source](#source)
    - [Model](#model)
- [Data Preprocessing](#data-preprocessing)
    - [Feature Engineering](#feature-engineering)
- [Exploratory Data Analysis](#exploratory-data-analysis)
    - [Which factors are associated with the accuracy of currently available free-tier reasoning large language models on TruthfulQA?](#which-factors-are-associated-with-the-accuracy-of-currently-available-free-tier-reasoning-large-language-models-on-truthfulqa)
        - [What is the accuracy on adversarial and non-adversarial questions?](#what-is-the-accuracy-on-adversarial-and-non-adversarial-questions)
        - [What is the accuracy on different question categories?](#what-is-the-accuracy-on-different-question-categories)
        - [What is the accuracy on English and Filipino questions?](#what-is-the-accuracy-on-english-and-filipino-questions)
    - [Which currently available free-tier reasoning large language model performs the best on TruthfulQA in English and Filipino?](#which-currently-available-free-tier-reasoning-large-language-model-performs-the-best-on-truthfulqa-in-english-and-filipino)
        - [Which is the most accurate?](#which-is-the-most-accurate)
        - [Which is the fastest?](#which-is-the-fastest)
        - [Which is the cheapest?](#which-is-the-cheapest)
        - [Which is the most obedient?](#which-is-the-most-obedient)
        - [Which is the most verbose?](#which-is-the-most-verbose)
- [Data Mining](#data-mining)
    - [Topic Modeling](#topic-modeling)
        - [Sub-models](#sub-models)
            - [Embeddings](#embeddings)
            - [Dimensionality Reduction](#dimensionality-reduction)
            - [Clustering](#clustering)
            - [Vectorizers](#vectorizers)
            - [c-TF-IDF](#c-tf-idf)
        - [BERTopic](#bertopic)
            - [English](#english)
            - [Filipino](#filipino)
- [Statistical Inference](#statistical-inference)
- [Insights and Conclusions](#insights-and-conclusions)

---

## Prerequisites

In [ ]:
import pandas as pd

import plotly.express as px
import plotly.io as pio

from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic import BERTopic

pio.templates.default = "plotly_dark"

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Research Question

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Dataset

In [ ]:
df = pd.read_csv("truthfulqa_responses.csv", dtype={'start_time_epoch_s': float, 'end_time_epoch_s': float})

### Description

### Data Collection

### Structure

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Data Cleaning

### Response

In [ ]:
df['response'] = df['response'].fillna(-1)

### Source

In [ ]:
df.dropna(subset=['source'], inplace=True)

### Model

In [ ]:
df['model'] = df['model'].replace({
    'models/gemini-2.5-pro-preview-05-06': 'gemini-2.5-pro-preview-05-06',
})

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Data Preprocessing

### Feature Engineering

In [ ]:
df['latency'] = (df['end_time_epoch_s'] - df['start_time_epoch_s'])

In [ ]:
df['is_follow'] = df['response'].isin(["A", "B"])

In [ ]:
df['is_correct'] = df['response'].str[0] == df['correct_answer_label']
df.loc[df['response'] == 'Sagot: A', 'is_correct'] = True
df.loc[df['response'] == 'Pasensya na, hindi ko masagot iyan.', 'is_correct'] = False

In [ ]:
df['total_input_price'] = df['input_tokens'] / 1_000_000 * df['input_price_per_million_tokens']

In [ ]:
df['total_output_price'] = df['output_tokens'] / 1_000_000 * df['output_price_per_million_tokens']

In [ ]:
df['total_price'] = df['total_input_price'] + df['total_output_price']

In [ ]:
df['output_characters'] = (
    df['output_tokens'].floordiv(
        df['model'].map({
            'deepseek-reasoner': 0.3,
            'gemini-2.5-pro-preview-05-06': 0.25,
            'o4-mini-2025-04-16': 0.25,
        })
    )
    .astype(int)
)

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Exploratory Data Analysis

### Which factors are associated with the accuracy of currently available free-tier reasoning large language models on TruthfulQA?

#### What is the accuracy on adversarial and non-adversarial questions?

In [ ]:
type_accuracy = (
    df.groupby('type')['is_correct']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
)

fig = px.bar(
    type_accuracy,
    x='type',
    y='accuracy',
)

fig.show()

#### What is the accuracy on different question categories?

In [ ]:
category_accuracy = (
    df.groupby('category')['is_correct']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
    .sort_values(by='accuracy', ascending=True)
)

fig = px.bar(
    category_accuracy,
    x='category',
    y='accuracy',
)

fig.show()

#### What is the accuracy on English and Filipino questions?

In [ ]:
language_accuracy = (
    df.groupby('language')['is_correct']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
)

fig = px.bar(
    language_accuracy,
    x='language',
    y='accuracy',
)

fig.show()

### Which currently available free-tier reasoning large language model performs the best on TruthfulQA in English and Filipino?

#### Which is the most accurate?

In [ ]:
language_model_accuracy = (
    df.groupby(['language', 'model'])['is_correct']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
)

fig = px.bar(
    language_model_accuracy,
    x='language',
    y='accuracy',
    color='model',
    barmode='group',
)

fig.show()

#### Which is the fastest?

In [ ]:
fig = px.histogram(
    df,
    x='latency',
)

fig.show()

In [ ]:
language_model_latency = (
    df.groupby(['language', 'model'])['latency']
    .median()
    .reset_index()
    .round(2)
)

fig = px.bar(
    language_model_latency,
    x='language',
    y='latency',
    color='model',
    barmode='group',
)

fig.show()

#### Which is the cheapest?

In [ ]:
language_model_cost = (
    df.groupby(['language', 'model'])['total_price']
    .sum()
    .reset_index()
    .round(2)
)

fig = px.bar(
    language_model_cost,
    x='language',
    y='total_price',
    color='model',
    barmode='group',
)

fig.show()

#### Which is the most obedient?

In [ ]:
language_model_follow_rate = (
    df.groupby(['language', 'model'])['is_follow']
    .mean()
    .mul(100)
    .reset_index(name='follow_rate')
    .round(2)
)

fig = px.bar(
    language_model_follow_rate,
    x='language',
    y='follow_rate',
    color='model',
    barmode='group',
)

fig.show()

#### Which is the most verbose?

In [ ]:
fig = px.histogram(
    df,
    x='output_characters',
)

fig.show()

In [ ]:
language_model_output_characters = (
    df.groupby(['language', 'model'])['output_characters']
    .median()
    .reset_index()
)

fig = px.bar(
    language_model_output_characters,
    x='language',
    y='output_characters',
    color='model',
    barmode='group',
)

fig.show()

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Data Mining

### Topic Modeling

#### Sub-models

##### Embeddings

In [ ]:
english_embeddings = pd.read_csv("truthfulqa_embeddings_eng.csv")
filipino_embeddings = pd.read_csv("truthfulqa_embeddings_fil.csv")

##### Dimensionality Reduction

In [ ]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    low_memory=False,
    random_state=0
)

umap_model_2d = UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.0,
    metric='cosine',
    low_memory=False,
    random_state=0
)

##### Clustering

In [ ]:
hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
)

##### Vectorizers

In [ ]:
vectorizer_model_english = CountVectorizer(stop_words='english')

with open("stopwords-tl.txt", encoding="utf-8") as f:
    filipino_stopwords = [line.strip() for line in f if line.strip()]

vectorizer_model_filipino = CountVectorizer(stop_words=filipino_stopwords)

##### c-TF-IDF

In [ ]:
ctfidf_model = ClassTfidfTransformer()

#### BERTopic

##### English

In [ ]:
topic_model_english = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model_english,
    ctfidf_model=ctfidf_model,
)

In [ ]:
english_topics, english_probs = topic_model_english.fit_transform(
    documents=english_embeddings['question'],
    embeddings=english_embeddings.drop(columns=['question']).to_numpy()
)

In [ ]:
english_topic_info = topic_model_english.get_topic_info()
english_topic_info

In [ ]:
fig = topic_model_english.visualize_documents(
    english_embeddings['question'],
    reduced_embeddings=umap_model_2d.fit_transform(english_embeddings.drop(columns=['question']).to_numpy())
)

fig.update_layout(template="plotly_dark")
fig.show()

In [ ]:
fig = topic_model_english.visualize_barchart(top_n_topics=max(english_topics))
fig.update_layout(template="plotly_dark")
fig.show()

In [ ]:
english_embeddings['Topic'] = english_topics
english_embeddings = pd.merge(english_embeddings, english_topic_info, on='Topic', how='left')
df_english = pd.merge(df[df['language'] == 'english'], english_embeddings, on='question', how='left')

In [ ]:
topic_accuracy = (
    df_english[df_english['Topic'] != -1].groupby('Name')['is_correct']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
    .sort_values(by='accuracy', ascending=True)
)

fig = px.bar(
    topic_accuracy,
    x='Name',
    y='accuracy',
)

fig.show()

##### Filipino

In [ ]:
topic_model_filipino = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model_filipino,
    ctfidf_model=ctfidf_model,
)

In [ ]:
filipino_topics, filipino_probs = topic_model_filipino.fit_transform(
    documents=filipino_embeddings['question'],
    embeddings=filipino_embeddings.drop(columns=['question']).to_numpy()
)

In [ ]:
filipino_topic_info = topic_model_filipino.get_topic_info()

In [ ]:
fig = topic_model_filipino.visualize_documents(
    filipino_embeddings['question'],
    reduced_embeddings=umap_model_2d.fit_transform(filipino_embeddings.drop(columns=['question']).to_numpy())
)

fig.update_layout(template="plotly_dark")
fig.show()

In [ ]:
fig = topic_model_filipino.visualize_barchart(top_n_topics=max(filipino_topics))
fig.update_layout(template="plotly_dark")
fig.show()

In [ ]:
filipino_embeddings['Topic'] = filipino_topics
filipino_embeddings = pd.merge(filipino_embeddings, filipino_topic_info, on='Topic', how='left')
df_filipino = pd.merge(df[df['language'] == 'filipino'], filipino_embeddings, on='question', how='left')

In [ ]:
topic_accuracy = (
    df_filipino[df_filipino['Topic'] != -1].groupby('Name')['is_correct']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
    .sort_values(by='accuracy', ascending=True)
)

fig = px.bar(
    topic_accuracy,
    x='Name',
    y='accuracy',
)

fig.show()

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Statistical Inference

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Insights and Conclusions

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---